# TriHydrA: complete example (generated streamflow data)

This notebook runs the TriHydrA pipeline end to end using four synthetic, daily streamflow stations covering 2000–2009.

It demonstrates:

- wide-CSV observation and simulation inputs;
- Layer 1 intrinsic checks;
- Layer 2 hydrological signatures;
- series-to-series comparison;
- Layer 3 network context;
- interactive HTML diagnostics; and
- network and per-station NetCDF outputs.

`example_ground_truth.csv` records the few anomalies inserted for teaching; TriHydrA does **not** read that file.

## 1. Locate the package and import TriHydrA

The cell works whether Jupyter starts in the repository root or in this `example` folder. It does not contain a user-specific drive path.

In [ ]:
from pathlib import Path
import os
import sys

here = Path.cwd().resolve()
candidates = [here, *here.parents]
PROJECT_ROOT = next(
    (folder for folder in candidates if (folder / "pyproject.toml").is_file() and (folder / "trihydra").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the TriHydrA repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from trihydra import run_batch

EXAMPLE_DIR = PROJECT_ROOT / "example"
CONFIG_PATH = EXAMPLE_DIR / "trihydra_example.toml"
print(f"TriHydrA project: {PROJECT_ROOT}")
print(f"Example config:   {CONFIG_PATH}")

## 2. Preview the two wide-CSV inputs

Each file has one `date` column and one discharge column per station. Both inputs use `mm/day`, which is declared explicitly in the TOML file.

In [ ]:
import pandas as pd

observations = pd.read_csv(EXAMPLE_DIR / "data" / "example_observations.csv", parse_dates=["date"])
simulations = pd.read_csv(EXAMPLE_DIR / "data" / "example_simulations.csv", parse_dates=["date"])

display(observations.head())
display(simulations.head())
print("Observation shape:", observations.shape)
print("Simulation shape: ", simulations.shape)

## 3. Inspect the known inserted issues

This teaching table tells us what was deliberately changed. It lets us compare expected anomalies with TriHydrA's findings without altering the pipeline.

In [ ]:
ground_truth = pd.read_csv(EXAMPLE_DIR / "data" / "example_ground_truth.csv")
display(ground_truth)

## 4. Review the configuration

The example TOML enables Layers 1–3 and pairwise comparison. It requests **HTML and NetCDF only**: `write_text = false`, `write_netcdf = true`, and `html_mode = "all"`.

Edit the TOML—not the scientific Python files—to change a normal run.

In [ ]:
print(CONFIG_PATH.read_text(encoding="utf-8"))

### Switching the input format

The active example uses wide CSV:

```toml
format = "csv"
path = "data/example_observations.csv"
date_column = "date"
```

For a NetCDF station-by-time source, replace those fields with:

```toml
format = "netcdf"
path = "path/to/input.nc"
variable = "streamflow"
station_coordinate = "basin"
time_coordinate = "date"
```

For the trusted ECMWF/AIFL long-term result format:

```toml
format = "aifl_pickle"
path = "path/to/pickle_folder"
trusted = true
```

Only load pickle files from a trusted source: unpickling can execute code. Paths are resolved relative to the TOML file.

## 5. Run the complete pipeline

`run_batch()` is the same validated workflow used by the command-line interface. Generated files are written under `example/results/`.

In [ ]:
batch = run_batch(CONFIG_PATH)
print("Output directory:", batch.output_directory)

## 6. Check processing status and scientific summaries

In [ ]:
display(batch.manifest)
print("Completed station IDs:", list(batch.station_results))
display(batch.summary)

## 7. Inspect one in-memory station result

The object keeps the scientific results available for further Python analysis even though this example does not write TXT reports.

In [ ]:
station_id = "EXAMPLE_001"
station_result = batch.station_results[station_id]

display(station_result.summary)
print("Layer 1 available:", station_result.layer1 is not None)
print("Layer 2 available:", station_result.layer2 is not None)
print("Comparison available:", station_result.comparison is not None)
print("Layer 3 available:", station_result.layer3 is not None)

## 8. Open the generated interactive HTML diagnostics

Each completed station receives Layer 1, Layer 2, candidate-series, comparison, and Layer 3 pages when applicable. The dropdown selects one page to display inside the notebook.

In [ ]:
from IPython.display import IFrame, display

station_folder = batch.output_directory / station_id
html_files = sorted(station_folder.glob("*.html"))
print("Available reports:")
for path in html_files:
    print(" -", path.name)

# Change this filename to display another report listed above.
report_name = "layer1.html"
report_path = station_folder / report_name
if report_path.is_file():
    display(IFrame(src=report_path.as_uri(), width="100%", height=700))
else:
    print(f"Report not found: {report_path}")

## 9. Explore the network NetCDF first

The network file is the batch index: it shows which stations completed and provides network-wide diagnostic trigger counts.

In [ ]:
import xarray as xr

network_path = batch.output_directory / "trihydra_network_summary.nc"
with xr.open_dataset(network_path) as network:
    print(network)
    network_table = network.to_dataframe().reset_index()

display(network_table.head())

## 10. Open one station NetCDF and list its groups

The station-file root is intentionally the human-facing summary. Detailed results are separated into named groups.

In [ ]:
import netCDF4

station_path = batch.output_directory / "stations" / f"{station_id}.nc"
with xr.open_dataset(station_path) as root:
    station_summary = root.to_dataframe().reset_index()

display(station_summary)

with netCDF4.Dataset(station_path) as nc:
    groups = list(nc.groups)

print("Available groups:", groups)

## 11. Explore detailed NetCDF groups

Start with `flags` for concise review evidence, then inspect scientific metrics or thresholds only when needed.

In [ ]:
def open_group(group_name):
    return xr.open_dataset(station_path, group=group_name)

for group_name in ["flags", "layer1", "layer2", "comparison", "layer3", "thresholds"]:
    if group_name not in groups:
        continue

    with xr.open_dataset(station_path, group=group_name) as dataset:
        print(f"\n--- {group_name.upper()} ---")
        print(dataset)

        if dataset.data_vars:
            table = dataset.to_dataframe().reset_index().head(12)
            display(table)

## 12. Where the generated files are stored

```text
example/results/
├── trihydra_network_summary.nc
├── trihydra_run.log
├── stations/
│   └── <station_id>.nc
└── <station_id>/
    ├── layer1.html
    ├── layer2.html
    ├── candidate_layer1.html
    ├── candidate_layer2.html
    ├── comparison.html
    └── layer3.html
```

Try changing one setting at a time—for example, disable one Layer 1 check or change `html_mode`—then rerun the notebook. Delete `example/results/` first only when you intentionally want a clean output directory.

In [ ]:
print("Example complete. NetCDF files were closed automatically.")